# 手臂擺幅角度&盒需圖.ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import os
import math
import pandas as pd
import numpy as np
from google.colab import files
import io
import matplotlib.pyplot as plt


In [ ]:
def calculate_arm_angle(x_shoulder, y_shoulder, x_elbow, y_elbow, x_wrist, y_wrist):

    a_x = x_elbow - x_shoulder
    a_y = y_elbow - y_shoulder

    b_x = x_wrist - x_elbow
    b_y = y_wrist - y_elbow

    dot_product = a_x * b_x + a_y * b_y

    magnitude_a = math.hypot(a_x, a_y)
    magnitude_b = math.hypot(b_x, b_y)

    if magnitude_a == 0 or magnitude_b == 0:
        return 0


    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))


    return math.degrees(math.acos(cos_angle))


In [ ]:
def determine_walking_direction(df):
    """
    根據肩膀的水平位置判斷病患是向左走還是向右走，並返回行走方向。
    """

    x_shoulder_left = df['kpt_5'].astype(float)
    x_shoulder_right = df['kpt_6'].astype(float)


    directions = []
    for i in range(len(x_shoulder_left)):
        if x_shoulder_left[i] < x_shoulder_right[i]:
            directions.append("left")
        else:
            directions.append("right")
    return directions


In [ ]:
def calculate_arm_angle_for_direction(df, directions):
    arm_angles = []
    for i in range(len(directions)):
        if directions[i] == "right":

            x_shoulder = df['kpt_13'].astype(float)[i]
            y_shoulder = df['kpt_14'].astype(float)[i]
            x_elbow = df['kpt_15'].astype(float)[i]
            y_elbow = df['kpt_16'].astype(float)[i]
            x_wrist = df['kpt_17'].astype(float)[i]
            y_wrist = df['kpt_18'].astype(float)[i]
        else:

            x_shoulder = df['kpt_6'].astype(float)[i]
            y_shoulder = df['kpt_7'].astype(float)[i]
            x_elbow = df['kpt_8'].astype(float)[i]
            y_elbow = df['kpt_9'].astype(float)[i]
            x_wrist = df['kpt_10'].astype(float)[i]
            y_wrist = df['kpt_11'].astype(float)[i]


        angle = calculate_arm_angle(x_shoulder, y_shoulder, x_elbow, y_elbow, x_wrist, y_wrist)
        arm_angles.append(angle)

    return arm_angles


In [ ]:
category_arm_angles = {}


In [ ]:
for category in range(1, 5):
    print(f"請上傳第 {category} 類的CSV文件")
    uploaded = files.upload()


    for file_name in uploaded.keys():

        df = pd.read_csv(io.BytesIO(uploaded[file_name]))


        kpts_columns = df['kpts'].str.split(',', expand=True)


        for i in range(kpts_columns.shape[1]):
            kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)


        df = df.join(kpts_columns, rsuffix='_kpts')


        for col in kpts_columns.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')


        directions = determine_walking_direction(df)


        arm_angles = calculate_arm_angle_for_direction(df, directions)


        df['arm_angle'] = arm_angles


        if category not in category_arm_angles:
            category_arm_angles[category] = []
        category_arm_angles[category].extend(arm_angles)


        print(df[['kpt_13', 'kpt_14', 'kpt_15', 'kpt_16', 'kpt_17', 'kpt_18', 'arm_angle']].head())


In [ ]:
category_stats = {}
for category, angles in category_arm_angles.items():
    q1 = np.percentile(angles, 25)
    q2 = np.percentile(angles, 50)
    q3 = np.percentile(angles, 75)
    mean_angle = np.mean(angles)
    category_stats[category] = {
        'Q1': q1,
        'Median (Q2)': q2,
        'Q3': q3,
        'Mean': mean_angle
    }


In [ ]:
all_angles = [angle for angles in category_arm_angles.values() for angle in angles]
total_mean_angle = np.mean(all_angles)


In [ ]:
for category, stats in category_stats.items():
    print(f"Category {category} 統計結果：")
    print(f"  Q1: {stats['Q1']:.2f}")
    print(f"  Median (Q2): {stats['Median (Q2)']:.2f}")
    print(f"  Q3: {stats['Q3']:.2f}")
    print(f"  Mean: {stats['Mean']:.2f}\n")

print(f"總平均角度：{total_mean_angle:.2f}\n")


In [ ]:
plt.figure(figsize=(10, 6))


In [ ]:
data = [category_arm_angles[cat] for cat in range(1, 5)]


In [ ]:
plt.boxplot(data, patch_artist=True, boxprops=dict(facecolor='skyblue'), showfliers=False)
plt.title('Comparison of Arm Angles for All Categories', fontsize=16)
plt.xlabel('Category', fontsize=14)
plt.ylabel('Arm Angle (Degrees)', fontsize=14)
plt.xticks([1, 2, 3, 4], ['Category 1', 'Category 2', 'Category 3', 'Category 4'])
plt.grid(True)


In [ ]:
plt.tight_layout()
plt.show()
